In [0]:
from pyspark.sql import functions as F

users_path = "/Volumes/credit_card_project/bronze/raw_data/users/sd254_users.csv"
cards_path = "/Volumes/credit_card_project/bronze/raw_data/cards/sd254_cards.csv"

users = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(users_path)
)

cards = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(cards_path)
)

print(f"Users: {users.count():,} rows")
print(f"Cards: {cards.count():,} rows")


First, read users and generate key:

In [0]:
import pandas as pd

users_pd = pd.read_csv(
    "/Volumes/credit_card_project/bronze/raw_data/users/sd254_users.csv"
)

users_pd.insert(
    0,
    "user_id",
    range(len(users_pd))
)

display(users_pd.head(10))


In [0]:
print("Number of users:", len(users_pd))
print("First user:", users_pd.iloc[0]["Person"])
print("Last user ID:", users_pd.iloc[-1]["user_id"])


In [0]:
users_pd = users_pd.rename(columns={
    "Person": "person",
    "Current Age": "current_age",
    "Retirement Age": "retirement_age",
    "Birth Year": "birth_year",
    "Birth Month": "birth_month",
    "Gender": "gender",
    "Address": "address",
    "Apartment": "apartment",
    "City": "city",
    "State": "state",
    "Zipcode": "zipcode",
    "Latitude": "latitude",
    "Longitude": "longitude",
    "Per Capita Income - Zipcode": "per_capita_income",
    "Yearly Income - Person": "yearly_income",
    "Total Debt": "total_debt",
    "FICO Score": "fico_score",
    "Num Credit Cards": "num_credit_cards"
})


Write users.parquet:

In [0]:
users_output_path = "/Volumes/credit_card_project/bronze/raw_data/users/users.parquet"

users_pd.to_parquet(
    users_output_path,
    index=False
)

print(f"Written: {users_output_path}")


In [0]:
users = spark.read.parquet(
    "/Volumes/credit_card_project/bronze/raw_data/users/users.parquet"
)

print(users.columns)


In [0]:
users_check = spark.read.parquet(users_output_path)

users_check.printSchema()
display(users_check.limit(10))


Second, read cards:

In [0]:
cards_pd = pd.read_csv(
    "/Volumes/credit_card_project/bronze/raw_data/cards/sd254_cards.csv"
)

print("Number of cards:", len(cards_pd))
print("Columns:", cards_pd.columns.tolist())

display(cards_pd.head(10))


In [0]:
cards_pd = cards_pd.rename(columns={
    "User": "user_id",
    "CARD INDEX": "card_id"
})

display(cards_pd.head(10))


In [0]:
print("Number of cards:", len(cards_pd))
print("Unique users:", cards_pd["user_id"].nunique())
print("User range:", cards_pd["user_id"].min(), "to", cards_pd["user_id"].max())
print("Card ID range:", cards_pd["card_id"].min(), "to", cards_pd["card_id"].max())


In [0]:
duplicate_keys = cards_pd.duplicated(
    subset=["user_id", "card_id"]
).sum()

print("Duplicate (user_id, card_id) keys:", duplicate_keys)


In [0]:
cards_pd = cards_pd.drop(
    columns=["Card Number", "CVV"]
)

display(cards_pd.head(10))


In [0]:
print("Columns remaining:")
print(cards_pd.columns.tolist())


In [0]:
#write 
cards_output_path = "/Volumes/credit_card_project/bronze/raw_data/cards/cards.parquet"

cards_pd.to_parquet(
    cards_output_path,
    index=False
)

print(f"Written: {cards_output_path}")


In [0]:
cards_check = spark.read.parquet(cards_output_path)

cards_check.printSchema()
display(cards_check.limit(10))


In [0]:
#rename columns
cards_pd = cards_pd.rename(columns={
    "Card Brand": "card_brand",
    "Card Type": "card_type",
    "Expires": "expires",
    "Has Chip": "has_chip",
    "Cards Issued": "cards_issued",
    "Credit Limit": "credit_limit",
    "Acct Open Date": "acct_open_date",
    "Year PIN last Changed": "year_pin_last_changed",
    "Card on Dark Web": "card_on_dark_web"
})


In [0]:
#write again
cards_output_path = "/Volumes/credit_card_project/bronze/raw_data/cards/cards.parquet"

cards_pd.to_parquet(
    cards_output_path,
    index=False
)

print(f"Written: {cards_output_path}")


In [0]:
cards_check = spark.read.parquet(cards_output_path)

cards_check.printSchema()
display(cards_check.limit(10))


In [0]:
cards = spark.read.parquet(
    "/Volumes/credit_card_project/bronze/raw_data/cards/cards.parquet"
)

print(cards.columns)

